# पाठ 10 - उत्पादनातील AI एजंट्स

या धड्यात तुम्ही Microsoft Agent Framework (Python) वापरून AI एजंट्ससाठीचे **उत्पादन नमुने** शिकाल. आपण समाविष्ट करतो:

- **निगराणी** — एजंट संवादांमध्ये वेळ मोजणे आणि लॉगिंग जोडणे
- **मूल्यांकन** — प्रतिसादाच्या गुणवत्तेचे मूल्यांकन करण्यासाठी एका मूल्यांकन एजंटचा वापर
- **खर्च व्यवस्थापन** — टोकन अनुकूलन आणि मॉडेल निवडीसाठी रणनीती

परिदृश्य हे एक **प्रवास एजंट** आहे जे वापरकर्त्यांना प्रवासाची योजना करण्यास मदत करते, ज्यावर निगराणी आणि मूल्यांकन लागू केलेले आहे.


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## उत्पादनातील विचार

AI एजंट्सना प्रोटोटाइपमधून उत्पादनात हलवताना तीन स्तंभांवर काळजीपूर्वक लक्ष देणे आवश्यक आहे:

1. **निरीक्षणक्षमता** — आपल्याला पाहण्याची क्षमता असावी की एजंट काय करत आहे, त्यास किती वेळ लागतो आणि तो कोणती साधने कॉल करतो. ट्रेसिंग आणि लॉगिंग शिवाय, उत्पादनातील समस्या डिबग करणे जवळजवळ अशक्य आहे.

2. **मूल्यांकन** — स्वयंचलित गुणवत्ता तपासण्या सुनिश्चित करतात की एजंटची उत्तरे कालांतराने अचूक, संपूर्ण आणि उपयुक्त राहतात. एक मूल्यांकन एजंट परिभाषित निकषांनुसार उत्तरे गुणांकन करू शकतो.

3. **खर्च व्यवस्थापन** — टोकन वापर थेट खर्चावर परिणाम करतो. प्रॉम्प्ट अनुकूलन, मॉडेल निवड आणि कॅशिंग सारख्या धोरणांनी खर्च गुणवत्तेची तडजोड न करता नियंत्रणात ठेवण्यास मदत होते.


## निरीक्षणीय एजंट तयार करणे

आम्ही प्रवास साधने परिभाषित करतो आणि एजंट कॉलला वेळ मोजणीने वेढून ठेवतो जेणेकरून आपण विलंब (latency) निरीक्षण करू शकू. उत्पादनात आपण OpenTelemetry किंवा तत्सम ट्रेसिंग बॅकएंडसोबत एकत्रित कराल.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन नमुने

एक सामान्य उत्पादन पद्धत म्हणजे दुसऱ्या एजंटचा वापर **मूल्यांकनकर्ता** म्हणून करणे. मूल्यांकनकर्ता प्राथमिक एजंटच्या प्रतिक्रियेला पूर्वनिर्धारित निकषांनुसार गुण देतो, जसे की पूर्णत्व, अचूकता आणि उपयोगीपणा.

हे सक्षम करते:
- उपयोगकर्त्यांपर्यंत उत्तरे पोहोचण्यापूर्वी स्वयंचलित गुणवत्ता नियंत्रणे
- प्रॉम्प्ट किंवा मॉडेल बदलल्यास रिग्रेशन शोधणे
- कालांतराने एजंटच्या कामगिरीचे सतत निरीक्षण


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## खर्च व्यवस्थापन धोरणे

उत्पादन AI एजंट्ससाठी खर्च नियंत्रण अत्यंत महत्त्वाचे आहे. येथे मुख्य धोरणे आहेत:

| Strategy | Description |
|---|---|
| **प्रॉम्प्ट अनुकूलन** | सिस्टम सूचना संक्षिप्त ठेवा. इनपुट टोकन्स कमी करण्यासाठी अनावश्यक संदर्भ काढा. |
| **मॉडेल निवड** | वर्गीकरण किंवा एक्सट्रॅक्शन सारखी सोपी कामे करण्यासाठी लहान, स्वस्त मॉडेल्स (उदा. GPT-4o-mini) वापरा, आणि जटिल तर्कसंग्रहासाठी मोठी मॉडेल राखून ठेवा. |
| **कॅशिंग** | टूल परिणाम आणि वारंवार विचारल्या जाणाऱ्या क्वेरीज कॅश करा जेणेकरून अनावश्यक API कॉल टाळता येतील. |
| **टोकन बजेट** | अनपेक्षितपणे लांब प्रतिसाद टाळण्यासाठी `max_tokens` मर्यादा सेट करा. |
| **बॅचिंग** | जिथे शक्य असेल, एकाधिक वापरकर्ता क्वेरी एका API कॉलमध्ये गटबद्ध करा. |

व्यवहारात, स्तरबद्ध पद्धत प्रभावी आहे: सरळ विनंत्या जलद, स्वस्त मॉडेलकडे मार्गदर्शित करा आणि फक्त जटिल क्वेरीज अधिक सक्षम मॉडेलकडे अग्रेषित करा.


## सारांश

या धड्यात तुम्ही हे कसे करायचे ते शिकलात:

1. **प्रेक्षणीयता जोडा** एजंट संवादांमध्ये टाइमिंग आणि लॉगिंगसह, ट्रेसिंग आणि मॉनिटरिंगसाठी पाया तयार करत.
2. **एजंटच्या प्रतिसादांचे मूल्यांकन करा** स्वयंचलितपणे अशा एका मूल्यांकन एजंटचा वापर करून जो पूर्णता, अचूकता आणि उपयुक्ततेला गुण देतो.
3. **खर्च व्यवस्थापित करा** प्रॉम्प्ट ऑप्टिमायझेशन, मॉडेल निवड, कॅशिंग, आणि टोकन बजेटद्वारे.

हे उत्पादन नमुने सुनिश्चित करतात की तुमचे AI एजंट मोठ्या प्रमाणावर विश्वासार्ह, मोजता येण्याजोगे, आणि खर्च-कुशल असतील.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, परंतु कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये त्रुटी किंवा चुका असू शकतात. मुळ भाषेतील मूळ दस्तऐवज अधिकृत स्रोत म्हणून मानले पाहिजे. महत्त्वाच्या माहितीसाठी व्यावसायिक मानवी अनुवाद शिफारसीय आहे. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थनिर्वचनांसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
